In [ ]:
# 1. Setup de caminhos locais
import os
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH:", RAW_PATH)


In [ ]:
# 2. Parametros globais e utilitarios
import pandas as pd
import yfinance as yf
from datetime import datetime

PERIODO = cfg.get_periodo_estudo()
START_DATE = PERIODO["start"]
END_DATE = PERIODO["end"]
TICKERS = {"ibovespa": "^BVSP", "bova11": "BOVA11.SA"}

os.makedirs(RAW_PATH, exist_ok=True)
print(f"Periodo configurado: {START_DATE} -> {END_DATE}")
print("Tickers alvo:", TICKERS)


In [ ]:
# 3. Funcoes para download via yfinance
from typing import Tuple
import subprocess
import sys

def download_ticker(name: str, ticker: str, start: str, end: str) -> Tuple[Path, pd.DataFrame]:
    print(f"Baixando {ticker} ({start} -> {end})...")
    try:
        data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
        if data.empty:
            raise ValueError(f"Sem dados retornados para {ticker} no periodo especificado.")
        data = data.reset_index().rename(columns={"Date": "day", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Adj Close": "adj_close", "Volume": "volume"})
        data["source_ticker"] = ticker
        target_path = Path(RAW_PATH) / f"{name}.csv"
        data.to_csv(target_path, index=False)
        print(f"Salvo em {target_path}")
        return target_path, data
    except Exception as e:
        print(f"ERRO ao baixar {ticker}: {e}")
        print(f"Tentando usar script de geracao de dados de amostra...")
        # Try to use the sample data generation script
        script_path = PROJECT_PATHS['repo_root'] / 'scripts' / 'generate_sample_data.py'
        if script_path.exists():
            subprocess.run([sys.executable, str(script_path)], check=True)
            # Load the generated file
            target_path = Path(RAW_PATH) / f"{name}.csv"
            if target_path.exists():
                data = pd.read_csv(target_path, parse_dates=['day'])
                print(f"Dados de amostra carregados de {target_path}")
                return target_path, data
        raise


In [ ]:
# 4. Executar downloads para todos os tickers
summary_rows = []
download_attempted = False
for name, ticker in TICKERS.items():
    if not download_attempted:
        # Try downloading the first ticker
        try:
            path, df = download_ticker(name, ticker, START_DATE, END_DATE)
            summary_rows.append({"arquivo": path.name, "ticker": ticker, "records": len(df), "inicio": df["day"].min(), "fim": df["day"].max()})
            download_attempted = True
        except:
            # If first download fails, sample data was generated for all tickers
            download_attempted = True
            # Load all generated files
            for fname, fticker in TICKERS.items():
                fpath = Path(RAW_PATH) / f"{fname}.csv"
                if fpath.exists():
                    fdf = pd.read_csv(fpath, parse_dates=['day'])
                    summary_rows.append({"arquivo": fpath.name, "ticker": fticker, "records": len(fdf), "inicio": fdf["day"].min(), "fim": fdf["day"].max()})
            break
    else:
        # Continue with remaining tickers
        path, df = download_ticker(name, ticker, START_DATE, END_DATE)
        summary_rows.append({"arquivo": path.name, "ticker": ticker, "records": len(df), "inicio": df["day"].min(), "fim": df["day"].max()})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


In [ ]:
# 5. Listar arquivos gerados em data_raw
for path in Path(RAW_PATH).glob('*.csv'):
    size_kb = path.stat().st_size / 1024
    print(f" - {path.name}: {size_kb:0.1f} KB")
